# JDsearch pipeline: run every stage

A thin wrapper around `scripts/`. Each cell runs one stage as a subprocess (same as the command line), so a crash in one
stage never leaves a half-initialised kernel. Stages skip work whose outputs exist; add `--force` to redo one.

Set `CONFIG` to `configs/smoke.yaml` for a 5-minute CPU check, `configs/default.yaml` for the real run.

In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
CONFIG = "configs/default.yaml"     # or "configs/smoke.yaml"
EXTRA = []                          # e.g. ["--set", "tower.epochs=2"]

def stage(script, *args):
    cmd = [sys.executable, f"scripts/{script}", "--config", CONFIG, *EXTRA, *args]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

## 0. Parse raw files, shuffle slates, split users, assign cross-fitting folds

In [ ]:
stage("00_prepare.py")

## 1. EDA: history retention windows, type mix by age, slates

In [ ]:
stage("01_eda.py")

## 2. Features, baselines, prior team's model

In [ ]:
stage("02_features.py")

In [ ]:
stage("02b_baselines.py")

In [ ]:
stage("02c_prior_ncf.py")

## 3–4. Two-tower: train cross-fitted models, then score every displayed item

In [ ]:
stage("03_train_tower.py")

In [ ]:
stage("04_score_tower.py")

## 5–6. LambdaRank re-ranker: train, then predict validation / test

In [ ]:
stage("05_train_ranker.py")

In [ ]:
stage("06_predict.py")

## 7. Diagnostics (add `ablations` to `--parts` for the feature-group ablations, ~10 extra ranker fits)

In [ ]:
stage("07_diagnostics.py", "--parts", "segments,shap,attention")